# Lorenz '96 model in JAX

The *Lorenz '96* model is an idealised dynamical system model formulated by Ed Lorenz in 1996 [1]. The formulation considered here is
$$
\frac{\mathrm{d}x_i}{\mathrm{d}t}=(x_{i+1}-x_{i-2})\:x_{i-1}-x_i+F,
$$
where $\mathbf{x}=(x_1,x_2,\dots,x_N)$ for some $N\geq4$ and we use the following conventions:

* $x_0=x_{-1}=x_{N-1}$
* $x_1=x_{N+1}$

We use the NumPy implementation found on the Wikipedia page [2] but make use of `jax.numpy` rather than standard NumPy.

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

Set model parameters

In [ ]:
N = 5  # Number of variables
F = 8  # Forcing

Define the RHS function

In [ ]:
def L96(x, t):
    """Lorenz 96 model with constant forcing"""
    return (jnp.roll(x, -1) - jnp.roll(x, 2)) * jnp.roll(x, 1) - x + F 

Define a perturbation

In [ ]:
epsilon = np.zeros(N)
epsilon[0] = 0.1
epsilon = jnp.asarray(epsilon)

Define the timestep and discretise the time window we seek to integrate over.

In [ ]:
dt = 0.01
end_time = 30.0
t = jnp.arange(0.0, end_time, dt)

Define initial condition

In [ ]:
x0 = F * jnp.ones(N)  # Initial state (equilibrium)

Integrate in time using explicit Euler

In [ ]:
def explicit_euler(epsilon):
    # TODO: Docs
    trajectory = [x0 + epsilon]  # Add small perturbation to the first variable
    for ti in t:
        x_ = trajectory[-1]
        x = x_ + L96(x_, ti) * dt
        trajectory.append(x)
    return jnp.array(trajectory)

Plot the first three variables on 3D axes

In [ ]:
trajectory = explicit_euler(epsilon)

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2])
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_zlabel("$x_3$")
plt.show()

In [ ]:
import jax

In [ ]:
e0 = np.zeros_like(epsilon)
e0[0] = 1.0
p, J = jax.jvp(explicit_euler, (epsilon,), (e0,))

Check the function evaluation give the same result

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot(p[:, 0], p[:, 1], p[:, 2])
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_zlabel("$x_3$")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot(J[:, 0], J[:, 1], J[:, 2])
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_zlabel("$x_3$")
plt.show()

## References

[1] Lorenz, Edward (1996). "Predictability – A problem partly solved" (PDF). Seminar on Predictability, Vol. I, ECMWF. https://www.ecmwf.int/sites/default/files/elibrary/1995/10829-predictability-problem-partly-solved.pdf

[2] https://en.wikipedia.org/wiki/Lorenz_96_model